# Importing libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# Data loading and processing

In [ ]:
path = "/content/Big_Black_Money_Dataset.csv"
df = pd.read_csv(path)
df.head()

In [3]:
df['Date of Transaction'] = pd.to_datetime(df['Date of Transaction']).dt.date
df.drop('Transaction ID', axis=1, inplace=True)

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Amount (USD),10000.0,2.501818e+06,1.424364e+06,10031.79673,1.279005e+06,2.501310e+06,3.722416e+06,4.999812e+06
Money Laundering Risk Score,10000.0,5.526400e+00,2.893603e+00,1.00000,3.000000e+00,6.000000e+00,8.000000e+00,1.000000e+01
Shell Companies Involved,10000.0,4.469400e+00,2.879773e+00,0.00000,2.000000e+00,4.000000e+00,7.000000e+00,9.000000e+00


In [ ]:
money_ilegal = df[df["Source of Money"] == "Illegal"]
money_ilegal

# Data visualization

In [6]:
def plot_money_ilegal(column_name):
  plt.figure(figsize=(12, 6))
  money_ilegal.groupby(column_name).size().sort_values().plot(
      kind='barh', color=sns.palettes.mpl_palette('Dark2'))
  plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
plot_money_ilegal("Industry")

In [ ]:
plot_money_ilegal("Country")

In [ ]:
plot_money_ilegal("Destination Country")

In [ ]:
plot_money_ilegal("Tax Haven Country")

In [ ]:
plot_money_ilegal("Transaction Type")

In [ ]:
# Create a directed graph
graph = nx.DiGraph()

# Iterate through the DataFrame and add edges for each transaction
for index, row in money_ilegal.iterrows():
  source_country = row['Country']
  destination_country = row['Destination Country']

  # Add edge with weight 1
  if graph.has_edge(source_country, destination_country):
    graph[source_country][destination_country]['weight'] += 1
  else:
    graph.add_edge(source_country, destination_country, weight=1)

# Plot the graph
plt.figure(figsize=(15, 10))
pos = nx.spring_layout(graph)
nx.draw(graph, pos, with_labels=True, node_color='skyblue', node_size=1500, edge_color='gray', width=2, font_size=10)
plt.title('Ligações entre Países de Origem e Destino de Dinheiro Ilegal')
plt.show()

# Prediction

In [13]:
df.drop('Date of Transaction', axis=1, inplace=True)
df["Source of Money"] = df["Source of Money"].map({"Illegal": 1, "Legal": 0})
df["Reported by Authority"] = df["Reported by Authority"].map({True: 1, False: 0})

In [14]:
features = df.drop('Source of Money', axis=1)
target = df['Source of Money'].to_numpy()
target = target.reshape(-1, 1)

In [15]:
encoder = LabelEncoder()
for column in features.columns:
  if features[column].dtype == 'object':
    features[column] = encoder.fit_transform(features[column])

In [16]:
mm = MinMaxScaler()
features = mm.fit_transform(features)

In [17]:
# Define input layer
input_layer = Input(shape=(features.shape[1],))

# Define the rest of the model
x = Dense(64, activation='relu')(input_layer)
x = Dropout(0.2)(x)
x = Dense(32, activation='relu')(x)
output_layer = Dense(1, activation='sigmoid')(x)

# Create the model
model = Model(inputs=input_layer, outputs=output_layer)

model.compile(loss=tf.keras.losses.BinaryCrossentropy(),
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              metrics=['mae', "mse"])

early_stopping = EarlyStopping(monitor='val_loss', patience=5)

history = model.fit(
    features, target, epochs=15, batch_size=32,
    validation_split=0.2, callbacks=[early_stopping]
)

Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.6337 - mae: 0.4416 - mse: 0.2209 - val_loss: 0.6151 - val_mae: 0.4230 - val_mse: 0.2120
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.6204 - mae: 0.4270 - mse: 0.2143 - val_loss: 0.6157 - val_mae: 0.4174 - val_mse: 0.2122
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.6025 - mae: 0.4120 - mse: 0.2060 - val_loss: 0.6156 - val_mae: 0.4253 - val_mse: 0.2122
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6090 - mae: 0.4193 - mse: 0.2092 - val_loss: 0.6158 - val_mae: 0.4265 - val_mse: 0.2123
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6152 - mae: 0.4247 - mse: 0.2120 - val_loss: 0.6166 - val_mae: 0.4208 - val_mse: 0.2126
Epoch 6/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6080 - mae: 0.4158 - mse: 0.2086 - val_loss: 0.6164 - val_mae: 0.4281 - val_mse: 0.2126


In [18]:
def plot_history(history, metric):
    plt.plot(history.history[metric])
    plt.plot(history.history['val_' + metric])
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend([metric, 'val_' + metric])
    plt.show()

In [ ]:
plot_history(history, "loss")
plot_history(history, "mae")
plot_history(history, "mse")